# 07 · Ablation Study

Compare the ML solver (checkpoint trained via `train.py`) against the
classical triangle-hash solver on the **same** held-out synthetic
test set. Metrics: mean and median great-circle angular separation,
% within 5°/1°, and median solve time per image.

Run after `train.py` has produced `checkpoints/best.pt`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import torch
from PIL import Image

from src.data.augmentations import build_eval_transforms
from src.data.catalog import load_hyg_catalog
from src.data.dataset import SyntheticStarFieldDataset
from src.data.renderer import render_star_field, sample_random_pointing
from src.models.astrolocnet import AstroLocNet
from src.models.classical_solver import ClassicalSolver
from src.training.trainer import Trainer
from src.utils.coordinates import angular_separation_deg

In [ ]:
CHECKPOINT = ROOT / 'checkpoints/best.pt'
assert CHECKPOINT.exists(), 'Run `python train.py --config configs/default.yaml --smoke` first.'

catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=8.0)

model = AstroLocNet(pretrained=False)
Trainer.load_state(model, CHECKPOINT)
model.eval()

solver = ClassicalSolver(catalog, max_catalog_stars=300, max_triangle_arc_deg=20.0)
solver.build_index(verbose=True)
eval_tfm = build_eval_transforms(224)

## Generate a small shared test set and solve with both methods

In [ ]:
N = 25  # bump to 200+ for a real run
rng = np.random.default_rng(123)
rows = []
for i in range(N):
    ra, dec, rot, fw = sample_random_pointing(rng, (15.0, 80.0))
    img = render_star_field(ra, dec, fw, rot, catalog, image_size=224, rng=np.random.default_rng(1000 + i))
    img_u8 = (img * 255).astype(np.uint8)
    # ML
    pil = Image.fromarray(img_u8)
    t = time.perf_counter()
    with torch.no_grad():
        out = model(eval_tfm(pil).unsqueeze(0)).squeeze(0).numpy()
    ml_time = time.perf_counter() - t
    ml_sep = angular_separation_deg(ra, dec, float(out[0]) % 360.0, float(np.clip(out[1], -90, 90)))
    # Classical
    cl = solver.solve(img_u8)
    cl_sep = angular_separation_deg(ra, dec, cl.ra, cl.dec) if cl.success else np.nan
    rows.append({'ml_sep_deg': float(ml_sep), 'ml_time_s': ml_time,
                 'cl_sep_deg': float(cl_sep), 'cl_time_s': cl.solve_time, 'cl_success': cl.success})
df = pd.DataFrame(rows); df.head()

In [ ]:
def summary(s):
    s = s.dropna()
    return {'mean_deg': float(s.mean()), 'median_deg': float(s.median()),
            'pct_within_5': float((s <= 5).mean() * 100),
            'pct_within_1': float((s <= 1).mean() * 100)}
summary_df = pd.DataFrame({
    'ML': summary(df['ml_sep_deg']),
    'Classical': summary(df['cl_sep_deg']),
}).T
summary_df['median_time_ms'] = [df['ml_time_s'].median() * 1000, df['cl_time_s'].median() * 1000]
summary_df['cl_success_rate_%'] = ['—', f"{df['cl_success'].mean()*100:.0f}"]
print(summary_df.to_string())
summary_df.to_csv(ROOT / 'reports/ablation_results.csv')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['ml_sep_deg'], bins=20, alpha=0.7, color='#7c5cff', label='ML')
ax.hist(df['cl_sep_deg'].dropna(), bins=20, alpha=0.7, color='#ffd166', label='Classical')
ax.set_xlabel('Angular separation from truth (°)'); ax.set_ylabel('Count')
ax.set_title('ML vs Classical: per-image error distribution'); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/07_ablation_hist.png', dpi=150)
plt.show()